# NEXTBUY: Reorder model

The goal of this model is to predict whether a product is going to be reordered when bought.

### Part 1: Import cleaned feature engineered dataset

In [ ]:
import pandas as pd
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn import metrics





PROCESSED_PATH = Path("processed/")
parquet_file = PROCESSED_PATH / "full_data_engineered.parquet"
fallback_file = Path("full_data_engineered.parquet")

if parquet_file.exists():
    print("Loading from processed folder...")
    full_data = pd.read_parquet(parquet_file)
elif fallback_file.exists():
    print("Loading from current directory (fallback)...")
    full_data = pd.read_parquet(fallback_file)
else:
    raise FileNotFoundError(
        f"Processed data not found. "
        "Please run the feature engineering notebook first."
    )

print(f"Loaded: {full_data.shape}")

full_data.dropna(inplace=True)

In [ ]:
full_data

### Features preparation and splitting

In [ ]:
featuresList = ["add_to_cart_order", "order_number", "days_since_prior_order", "product_buys_count"]
X = full_data[featuresList]
y = full_data["reordered"]


Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=12)

Xtrain.head(), ytrain.head()

### Model training: Logistic Regression

In [ ]:
logreg = LogisticRegression()
logreg.fit(Xtrain, ytrain)

### Model training: Random Forest

In [ ]:
ranfor = RandomForestClassifier(n_estimators=100)
ranfor.fit(Xtrain, ytrain)

## Predictions and scores

### Confusion Matrix

In [ ]:
yPredictions = logreg.predict(Xtest)
yPredictionsRF = ranfor.predict(Xtest)

confMatrix = metrics.confusion_matrix(ytest, yPredictions)
report = metrics.classification_report(ytest, yPredictions, target_names=["not reordered", "reordered"])

confMatrixRF = metrics.confusion_matrix(ytest, yPredictionsRF)
reportRF = metrics.classification_report(ytest, yPredictionsRF, target_names=["not reordered", "reordered"])

print(confMatrix, "\n", report, "\n")
print(confMatrixRF, "\n", reportRF)

### ROC Curve

In [ ]:
yPredProba = logreg.predict_proba(Xtest)[:,1]

fpr, tpr, tresholds = metrics.roc_curve(ytest, yPredProba)
auc = metrics.roc_auc_score(ytest, yPredProba)

plt.plot(fpr, tpr, label=f"AUC={auc}")
plt.legend()
plt.show()



yPredProbaRF = ranfor.predict_proba(Xtest)[:,1]

fpr2, tpr2, tresholds2 = metrics.roc_curve(ytest, yPredProbaRF)
auc2 = metrics.roc_auc_score(ytest, yPredProbaRF)

plt.plot(fpr2, tpr2, label=f"AUC={auc2}")
plt.legend()
plt.show()

### Results visualization for each feature

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(10, 10))
ax = ax.flatten()

for i, feature in enumerate(featuresList):
    sns.scatterplot(x=Xtest[feature].values, y=yPredictions, hue=ytest, ax=ax[i])
    ax[i].set_xlabel(feature)
    ax[i].set_ylabel("Predicted LogReg")

plt.show()



fig2, ax2 = plt.subplots(nrows=2, ncols=2, figsize=(10, 10))
ax2 = ax2.flatten()

for i, feature in enumerate(featuresList):
    sns.scatterplot(x=Xtest[feature].values, y=yPredictionsRF, hue=ytest, ax=ax2[i])
    ax2[i].set_xlabel(feature)
    ax2[i].set_ylabel("Predicted RF")

plt.show()